In [1]:
import cv2 
import numpy as np

# ---------------------- 1. 图像读取与预处理 ----------------------
# 读取图像（替换为你的图片路径）
img = cv2.imread('torchModel/gear.bmp', cv2.IMREAD_GRAYSCALE)
if img is None:
    raise ValueError("无法读取图像，请检查路径")

# 高斯滤波去噪，减少轮廓检测的干扰
blur = cv2.GaussianBlur(img, (5, 5), 0)

# 自适应二值化，适配光照不均的情况
thresh = cv2.adaptiveThreshold(
    blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY_INV, 11, 2
)

# 形态学操作，去除小噪点，闭合齿轮轮廓
kernel = np.ones((3, 3), np.uint8)
morph = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)
morph = cv2.morphologyEx(morph, cv2.MORPH_OPEN, kernel, iterations=1)

# ---------------------- 2. 轮廓检测与筛选 ----------------------
# 提取所有轮廓
contours, hierarchy = cv2.findContours(
    morph, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
)

In [2]:
cv2.drawContours(img, contours, -1, (0, 255, 0), 2)

array([[80, 69, 73, ...,  0,  0,  0],
       [68, 76, 73, ...,  0,  0,  0],
       [62, 72, 78, ...,  0,  0,  0],
       ...,
       [61, 55, 61, ..., 63,  0,  0],
       [67, 61, 64, ...,  0,  0,  0],
       [62, 61, 53, ...,  0,  0,  0]], shape=(848, 744), dtype=uint8)

In [3]:
cv2.imshow("Contours", img)
cv2.waitKey(0)

-1

In [4]:
# 筛选齿轮轮廓（过滤过小/过大的噪点轮廓）
gear_contours = []
for cnt in contours:
    area = cv2.contourArea(cnt)
    # 面积阈值，根据你的图像尺寸调整（这里适配示例图）
    if 5000 < area < 50000:
        gear_contours.append(cnt)

# 按轮廓面积排序：最大的是齿圈，中间4个是行星轮，最小的是中心太阳轮
gear_contours = sorted(gear_contours, key=cv2.contourArea, reverse=True)
ring_gear = gear_contours[0]  # 最外层齿圈
planet_gears = gear_contours[1:5]  # 4个行星轮
#sun_gear = gear_contours[5]  # 中心太阳轮（如果存在）

# ---------------------- 3. 齿轮参数计算 ----------------------
def calculate_gear_params(contour, gear_name):
    """计算单个齿轮的尺寸参数"""
    # 1. 外接圆（齿顶圆）参数
    (x, y), radius = cv2.minEnclosingCircle(contour)
    center = (int(x), int(y))
    tip_diameter = 2 * radius  # 齿顶圆直径（齿轮总大小）
    
    # 2. 内接圆（齿根圆）参数（通过凸包近似）
    hull = cv2.convexHull(contour, returnPoints=True)
    (hx, hy), hull_radius = cv2.minEnclosingCircle(hull)
    root_diameter = 2 * hull_radius  # 齿根圆直径
    
    # 3. 齿数计算（通过轮廓的角点/凸缺陷计数）
    # 方法1：凸缺陷计数（适合齿轮齿形）
    hull_idx = cv2.convexHull(contour, returnPoints=False)
    defects = cv2.convexityDefects(contour, hull_idx)
    tooth_count = 0
    if defects is not None:
        for i in range(defects.shape[0]):
            s, e, f, d = defects[i, 0]
            # 深度阈值，过滤非齿形的凹陷
            if d > 1000:
                tooth_count += 1
    
    # 4. 轮廓周长、面积
    perimeter = cv2.arcLength(contour, True)
    area = cv2.contourArea(contour)
    
    # 打印参数
    print(f"=== {gear_name} 参数 ===")
    print(f"齿顶圆直径: {tip_diameter:.2f} 像素")
    print(f"齿根圆直径: {root_diameter:.2f} 像素")
    print(f"齿数: {tooth_count}")
    print(f"轮廓周长: {perimeter:.2f} 像素")
    print(f"轮廓面积: {area:.2f} 像素²\n")
    
    return center, radius, tip_diameter, tooth_count

# 计算齿圈参数
ring_center, ring_radius, ring_tip_dia, ring_teeth = calculate_gear_params(ring_gear, "齿圈")


=== 齿圈 参数 ===
齿顶圆直径: 388.91 像素
齿根圆直径: 388.91 像素
齿数: 6
轮廓周长: 2867.47 像素
轮廓面积: 17025.00 像素²



In [ ]:
cv2.circle(img, ring_center, int(ring_radius), (0, 0, 255), 2)
cv2.imshow("img", img)
cv2.waitKey(0)